In [1]:
import os
import yaml
import json
import hashlib
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from uuid import uuid4

# ML 라이브러리
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    confusion_matrix,
    roc_curve,
    classification_report,
)
import lightgbm as lgb

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

print("✅ 라이브러리 임포트 완료")

✅ 라이브러리 임포트 완료


In [2]:
# MLFLOW 
import mlflow
import mlflow.lightgbm

In [3]:
# callback
def mlflow_callback_from_grok():
    def _callback(env):
        # 평가 메트릭 가져오기
        if 'validation' in env.evaluation_result_list:
            for metric_name, value, *_ in env.evaluation_result_list:
                mlflow.log_metric(metric_name, value, step=env.iteration)
    return _callback


# MLflow LightGBM 콜백 함수
def mlflow_callback_from_gtp(env):
    """LightGBM 학습 과정에서 검증 성능을 MLflow에 로깅하는 콜백 함수"""
    for valid_set, valid_name in zip(env.evaluation_result_list, env.model_.valid_names_):
        metric_name, score, _ = valid_set  # (metric, value, is_higher_better)
        mlflow.log_metric(f"{valid_name}_{metric_name}", score, step=env.iteration)

In [4]:
# 디렉토리 설정
BASE_DIR = Path.cwd()
CONF_DIR = BASE_DIR / 'conf'
DATA_DIR = BASE_DIR / 'data'

def load_yaml(filepath: Path) -> dict:
    with open(filepath, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)

def save_yaml(data: dict, filepath: Path):
    with open(filepath, 'w', encoding='utf-8') as f:
        yaml.dump(data, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

# 설정 파일 로드
env_config = load_yaml(CONF_DIR / 'env.yml')
meta_config = load_yaml(CONF_DIR / 'meta.yml')
model_config = load_yaml(CONF_DIR / 'model.yml')

print("✅ 설정 파일 로드 완료")
print(f"   Project: {meta_config['project']}")
print(f"   User Id: {meta_config['user_id']}")
print(f"   Experiment: {meta_config['experiment']}")
print(f"   Algorithm: {model_config['algorithm']['name']}")

✅ 설정 파일 로드 완료
   Project: titanic-survival-prediction
   User Id: kunops
   Experiment: kunops-v1
   Algorithm: lightgbm


In [5]:
env_config

{'env': 'dev',
 'region': 'us-west-1',
 's3': {'conf_bucket': 'gs-retail-awesome-conf-us-west-1',
  'data_bucket': 'gs-retail-awesome-data-us-west-1',
  'model_bucket': 'gs-retail-awesome-model-us-west-1'},
 's3_paths': {'conf': '{env}/{user_id}/{project}/{experiment}/',
  'data': '{env}/{user_id}/{project}/{version}/',
  'model': '{env}/{user_id}/{project}/{experiment}/{run_id}/'},
 'local': {'root': 'run_pm',
  'input_dir': 'input',
  'output_dir': 'output',
  'output_subdirs': ['metadata',
   'config',
   'data_refs',
   'artifacts/model',
   'artifacts/metrics',
   'artifacts/charts',
   'artifacts/explainability',
   'reports']}}

In [6]:
meta_config

{'user_id': 'kunops',
 'project': 'titanic-survival-prediction',
 'experiment': 'kunops-v1',
 'model': 'passenger-survival-classifier',
 'version': 'v1.0',
 'notebook': 'titanic_modeling_mlflow.ipynb',
 'mlflow': {'tracking_uri': 'http://edu-mlflow-alb-697692344.us-east-1.elb.amazonaws.com',
  'experiment_name': 'titanic-survival-prediction/kunops-v1',
  'run_name': None,
  'username': 'edu01',
  'secret_name': 'gs-retail-mlflow-kunops',
  'region_name': 'us-west-1',
  'tags': ['lightgbm-classifier', 'titanic-kaggle', 'binary-classification']}}

In [7]:
model_config

{'algorithm': {'name': 'lightgbm',
  'task': 'classification',
  'suffix': 'kunops'},
 'hyperparameters': {'objective': 'binary',
  'boosting': 'gbdt',
  'max_depth': 6,
  'learning_rate': 0.05,
  'num_leaves': 31,
  'feature_fraction': 0.8,
  'bagging_fraction': 0.8,
  'bagging_freq': 5,
  'verbosity': 1,
  'num_iterations': 200,
  'min_data_in_leaf': 20,
  'metric': ['auc', 'binary_logloss', 'binary_error']},
 'data': {'seed': 42, 'train_valid_split': 0.2, 'stratify': True},
 'features': {'index_col': 'PassengerId',
  'target_col': 'Survived',
  'numeric_col': ['Age', 'Fare', 'SibSp', 'Parch'],
  'categorical_col': ['Pclass', 'Sex', 'Embarked'],
  'drop_col': ['Name', 'Ticket', 'Cabin']},
 'preprocessing': {'missing_values': {'Age': 'median',
   'Embarked': 'mode',
   'Fare': 'median'},
  'encoding': {'Sex': 'label', 'Embarked': 'onehot', 'Pclass': 'ordinal'},
  'scaling': {'method': 'standard', 'columns': ['Age', 'Fare']}},
 'feature_engineering': {'derived_features': [{'name': 'Fam

In [8]:
# meta.yml에서 MLflow 설정 로드
mlflow_config = meta_config.get('mlflow', {})
MLFLOW_TRACKING_URI = mlflow_config.get('tracking_uri')
MLFLOW_EXPERIMENT = mlflow_config.get('experiment_name')
MLFLOW_USERNAME = mlflow_config.get('username', '')
MLFLOW_TAGS = mlflow_config.get('tags', [])

In [9]:
# MLflow 인증 설정
os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = "edu01Pass!234"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

<Experiment: artifact_location='s3://retail-mlflow-artifacts/mlflow/artifacts/4', creation_time=1773797066952, experiment_id='4', last_update_time=1773797066952, lifecycle_stage='active', name='titanic-survival-prediction/kunops-v1', tags={'mlflow.experimentKind': 'custom_model_development'}, workspace='default'>

In [10]:
print("ENV:", os.getenv("MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING"))

mlflow.enable_system_metrics_logging()

print("ENV:", os.getenv("MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING"))

ENV: None
ENV: True


In [11]:
from datetime import datetime
from zoneinfo import ZoneInfo


def get_kst_timestamp() -> str:
    return datetime.now(ZoneInfo("Asia/Seoul")).strftime("%y%m%d%H%M%S")


kst_timestamp = get_kst_timestamp()
print(kst_timestamp)

260326163845


In [12]:
user_id = meta_config['user_id']
# MLflow Run 시작 (노트북 전체에서 하나의 run 사용)
mlflow_run = mlflow.start_run(run_name=f'{MLFLOW_EXPERIMENT}-{user_id}-{kst_timestamp}')

2026/03/26 07:38:45 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/03/26 07:38:45 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


In [13]:
print(f"✅ MLflow 연결 완료")
print(f"   Tracking URI:  {MLFLOW_TRACKING_URI}")
print(f"   Experiment:    {MLFLOW_EXPERIMENT}")
print(f"   Username:      {MLFLOW_USERNAME}")
print(f"   MLflow Run ID: {mlflow_run.info.run_id}")

✅ MLflow 연결 완료
   Tracking URI:  http://edu-mlflow-alb-697692344.us-east-1.elb.amazonaws.com
   Experiment:    titanic-survival-prediction/kunops-v1
   Username:      edu01
   MLflow Run ID: 1d20c0e810b1456487afa3da6a41518e


In [14]:
mlflow.set_tag("model_type", model_config['algorithm']['task'])
mlflow.set_tag("model_name", model_config['algorithm']['name'])

In [15]:
# mlflow.set_tag("tags", str(conf['MLFLOW'].get('tags', [])))
mlflow.set_tag("dataset_shape", "(1000,100)")
mlflow.set_tag("target_col", model_config['features']['target_col'])

In [16]:
df_shape_info = {"a":(1000,100)}
mlflow.log_params(df_shape_info)

In [17]:
params = model_config['hyperparameters']
boosting = model_config['class_weight']
mlflow.log_params(params)
mlflow.log_params(boosting)

In [18]:
mlflow.log_params({'imbalanced_info': model_config['class_weight']})

In [19]:
# if is_mlflow:
#     for index, row in metric_df.iterrows():
#         metric_name = index
#         mlflow.log_metric(f"{metric_name.lower()}/train", row['train_metrics'])       # Train 메트릭
#         mlflow.log_metric(f"{metric_name.lower()}/validation", row['valid_metric'])  # Validation 메트릭
#         mlflow.log_metric(f"{metric_name.lower()}/test", row['test_metric'])

In [20]:
import time
for i in range(1,100):
    time.sleep(2)

In [21]:
mlflow.end_run()

🏃 View run titanic-survival-prediction/kunops-v1-kunops-260326163845 at: http://edu-mlflow-alb-697692344.us-east-1.elb.amazonaws.com/#/experiments/4/runs/1d20c0e810b1456487afa3da6a41518e
🧪 View experiment at: http://edu-mlflow-alb-697692344.us-east-1.elb.amazonaws.com/#/experiments/4


2026/03/26 07:42:13 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/03/26 07:42:14 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!
